In [ ]:
# Analyse du marché IT au Maroc – Requêtes DuckDB

Ce notebook répond aux 5 questions analytiques du projet Data Lake.


In [ ]:

df_ent = pd.read_parquet('data_lake_root/gold/entreprises_recruteurs.parquet')
top10 = df_ent.nlargest(10, 'nb_offres_publiees')[['entreprise', 'nb_offres_publiees']].sort_values('nb_offres_publiees')

plt.figure(figsize=(10,6))
top10.plot(kind='barh', x='entreprise', y='nb_offres_publiees', legend=False, color='green')
plt.xlabel('Nombre d’offres publiées')
plt.title('Top 10 entreprises recruteuses (IT Maroc)')
plt.tight_layout()
plt.savefig('top_entreprises.png')
plt.show()

In [ ]:
import duckdb
con = duckdb.connect()

In [ ]:

## Question 1 : Quelles compétences sont les plus demandées au Maroc en IT ?

In [ ]:
SELECT
    famille,
    competence,
    nb_offres_mentionnent,
    pct_offres_total,
    rang_dans_profil
FROM read_parquet('data_lake_root/gold/top_competences.parquet')
WHERE profil = 'tous'     -- agrégat global si disponible
ORDER BY nb_offres_mentionnent DESC
LIMIT 20;

famille      │ competence │ nb_offres_total │
│     varchar      │  varchar   │     int128      │
├──────────────────┼────────────┼─────────────────┤
│ langages         │ sql        │            1350 │
│ langages         │ python     │            1197 │
│ cloud            │ azure      │             942 │
│ cloud            │ aws        │             876 │
│ langages         │ javascript │             856 │
│ langages         │ r          │             714 │
│ cloud            │ gcp        │             626 │
│ langages         │ java       │             542 │
│ frameworks_web   │ react      │             241 │
│ frameworks_web   │ angular    │             190 │
│ frameworks_web   │ spring     │             157 │
│ bi_analytics     │ tableau    │             136 │
│ frameworks_web   │ django     │             134 │
│ data_engineering │ kafka      │             129 │
│ data_engineering │ spark      │             124 │
│ data_engineering │ hadoop     │             110 │
│ bi_analytics     │ power_bi   │             107 │
│ data_engineering │ dbt        │              92 │
│ data_engineering │ airflow    │              91 │
└──────────────────┴────────────┴─────────────────
Interprétation:
SQL arrive en tête avec 1350 offres, juste devant Python (1197). Ce résultat montre que la maîtrise des bases de données et du langage SQL reste 
fondamentale pour la majorité des postes IT au Maroc. Python, bien que très présent, est légèrement moins cité que SQL, ce qui peut surprendre mais
s’explique par la forte demande sur des profils non développeurs (data analysts, BI) où SQL est indispensable.

Les technologies cloud sont très demandées : Azure (942), AWS (876) et GCP (626) figurent parmi les compétences les plus recherchées, dépassant même
des langages classiques comme Java (542) ou JavaScript (856). Cela traduit une accélération de la migration vers le cloud dans les entreprises 
marocaines.

On note une présence significative de R (714), bien au‑delà des attentes. Cela peut indiquer une forte activité en statistique et data science dans le
marché marocain, ou un biais lié aux sources d’offres (ex: secteurs académique ou financier). Les frameworks web (React, Angular, Spring, Django) sont
moins cités que les compétences cloud et langages de base, ce qui suggère que les offres généralistes privilégient les savoirs transverses plutôt que 
des frameworks très spécifiques.

Dans le domaine data engineering, Kafka (129), Spark (124) et Hadoop (110) sont les plus demandés, tandis que Airflow (91) et dbt (92) émergent. Les 
outils de BI (Tableau 136, Power BI 107) confirment l’importance de la visualisation et du reporting.

Conclusion : Un profil IT au Maroc doit maîtriser SQL, Python et au moins un cloud majeur (Azure ou AWS). Les compétences spécialisées (Spark, Kafka, 
Tableau) sont des plus‑values pour les postes techniques

In [ ]:
SELECT
    profil,
    famille,
    competence,
    nb_offres_mentionnent,
    rang_dans_profil
FROM read_parquet('data_lake_root/gold/top_competences.parquet')
WHERE profil IN ('Data Engineer', 'Data Analyst', 'Data Scientist')
  AND rang_dans_profil <= 5
ORDER BY profil, rang_dans_profil;

┌────────────────┬──────────────┬────────────┬───────────────────────┬──────────────────┐
│     profil     │   famille    │ competence │ nb_offres_mentionnent │ rang_dans_profil │
│    varchar     │   varchar    │  varchar   │         int64         │      int64       │
├────────────────┼──────────────┼────────────┼───────────────────────┼──────────────────┤
│ Data Analyst   │ langages     │ sql        │                   218 │                1 │
│ Data Analyst   │ langages     │ r          │                   151 │                2 │
│ Data Analyst   │ langages     │ python     │                   130 │                3 │
│ Data Analyst   │ cloud        │ aws        │                    69 │                4 │
│ Data Analyst   │ bi_analytics │ tableau    │                    65 │                5 │
│ Data Engineer  │ langages     │ sql        │                    57 │                1 │
│ Data Engineer  │ langages     │ python     │                    47 │                2 │
│ Data Engineer  │ langages     │ r          │                    45 │                3 │
│ Data Engineer  │ cloud        │ gcp        │                    21 │                4 │
│ Data Engineer  │ cloud        │ aws        │                    20 │                5 │
│ Data Scientist │ langages     │ sql        │                   193 │                1 │
│ Data Scientist │ langages     │ r          │                   133 │                2 │
│ Data Scientist │ langages     │ python     │                   119 │                3 │
│ Data Scientist │ cloud        │ azure      │                    66 │                4 │
│ Data Scientist │ cloud        │ gcp        │                    54 │                5 │
└────────────────┴──────────────┴────────────┴───────────────────────┴──────────────────┘
  15 rows                                                                     5 columns

In [ ]:
Interprétation:
SQL est la compétence numéro 1 dans les trois profils (Data Analyst, Data Engineer, Data Scientist). Cela confirme que la maîtrise des bases de données
et du langage d’interrogation est absolument fondamentale pour tous les métiers de la data au Maroc.

Python figure dans le top 3 des trois profils, mais arrive seulement en 3ème position chez Data Analyst (derrière SQL et R) et en 2ème chez Data 
Engineer. R est très présent (2ème chez Data Analyst et Data Scientist, 3ème chez Data Engineer), ce qui peut refléter une forte demande en statistique
et analyse de données, peut-être dans des secteurs comme la finance ou la recherche.

Du côté du cloud, on observe des différences : Data Analyst utilise surtout AWS (4ème compétence), Data Engineer utilise GCP puis AWS, tandis que Data
Scientist utilise Azure puis GCP. Cela suggère que le choix du cloud dépend des missions spécifiques de chaque profil.

Les outils de BI (Tableau) apparaissent uniquement pour Data Analyst (5ème), ce qui est logique car ces profils sont souvent en charge de la 
visualisation et du reporting. En revanche, on ne voit pas Spark, Kafka ou Airflow dans ces tops 5 : ils arrivent plus loin dans le classement général
(d’après la question 1 globale).

In [1]:
import matplotlib.pyplot as plt
import pandas as pd

# Charger les données (tu peux aussi utiliser ton DataFrame resultat_q1)
df_comp = pd.read_parquet('data_lake_root/gold/top_competences.parquet')
top20 = df_comp.groupby('competence')['nb_offres_mentionnent'].sum().nlargest(20).sort_values()

plt.figure(figsize=(10, 8))
top20.plot(kind='barh', color='skyblue')
plt.xlabel('Nombre d’offres')
plt.title('Top 20 compétences IT au Maroc')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('top_competences.png')  # sauvegarde l'image
plt.show()

<class 'ModuleNotFoundError'>: No module named 'pandas'

In [ ]:

##Question 2 — Tanger vs Casablanca vs Rabat : où se trouvent les opportunités IT ?

In [ ]:
┌
────────────┬──────────────────────────┬───────────┬──────────────────┬────────────┬────────────┐
│   ville    │          profil          │ nb_offres │ nb_offres_remote │ pct_remote │ rang_ville │
│  varchar   │         varchar          │   int64   │      int64       │   double   │   int64    │
├────────────┼──────────────────────────┼───────────┼──────────────────┼────────────┼────────────┤
│ Casablanca │ Admin Systèmes & Réseaux │        11 │                5 │       45.5 │          1 │
│ Casablanca │ Admin Systèmes & Réseaux │        10 │                5 │       50.0 │          2 │
│ Casablanca │ Admin Systèmes & Réseaux │         8 │                7 │       87.5 │          3 │
│ Casablanca │ Admin Systèmes & Réseaux │         8 │                4 │       50.0 │          3 │
│ Casablanca │ Admin Systèmes & Réseaux │         8 │                4 │       50.0 │          3 │
│ Casablanca │ Admin Systèmes & Réseaux │         8 │                4 │       50.0 │          3 │
│ Casablanca │ Admin Systèmes & Réseaux │         8 │                4 │       50.0 │          3 │
│ Casablanca │ Admin Systèmes & Réseaux │         8 │                3 │       37.5 │          3 │
│ Casablanca │ Admin Systèmes & Réseaux │         7 │                4 │       57.1 │          9 │
│ Casablanca │ Admin Systèmes & Réseaux │         7 │                4 │       57.1 │          9 │
│ Rabat      │ Admin Systèmes & Réseaux │         6 │                2 │       33.3 │         11 │
│ Casablanca │ Admin Systèmes & Réseaux │         6 │                3 │       50.0 │         11 │
│ Casablanca │ Admin Systèmes & Réseaux │         6 │                2 │       33.3 │         11 │
│ Casablanca │ Admin Systèmes & Réseaux │         6 │                5 │       83.3 │         11 │
│ Casablanca │ Admin Systèmes & Réseaux │         6 │                4 │       66.7 │         11 │
│ Rabat      │ Admin Systèmes & Réseaux │         6 │                3 │       50.0 │         11 │
│ Casablanca │ Admin Systèmes & Réseaux │         5 │                3 │       60.0 │         17 │
│ Casablanca │ Admin Systèmes & Réseaux │         5 │                2 │       40.0 │         17 │
│ Casablanca │ Admin Systèmes & Réseaux │         5 │                4 │       80.0 │         17 │
│ Rabat      │ Admin Systèmes & Réseaux │         5 │                1 │       20.0 │         17 │
│  ·         │           ·              │         · │                · │         ·  │          · │
│  ·         │           ·              │         · │                · │         ·  │          · │
│  ·         │           ·              │         · │                · │         ·  │          · │
│ Fès        │ Développeur Full Stack   │         1 │                1 │      100.0 │         50 │
│ Rabat      │ Développeur Full Stack   │         1 │                0 │        0.0 │         50 │
│ Fès        │ Développeur Full Stack   │         1 │                1 │      100.0 │         50 │
│ Fès        │ Développeur Full Stack   │         1 │                0 │        0.0 │         50 │
│ Marrakech  │ Développeur Full Stack   │         1 │                1 │      100.0 │         50 │
│ Marrakech  │ Développeur Full Stack   │         1 │                1 │      100.0 │         50 │
│ Marrakech  │ Développeur Full Stack   │         1 │                0 │        0.0 │         50 │
│ Tanger     │ Développeur Full Stack   │         1 │                1 │      100.0 │         50 │
│ Rabat      │ Développeur Full Stack   │         1 │                1 │      100.0 │         50 │
│ Marrakech  │ Développeur Full Stack   │         1 │                0 │        0.0 │         50 │
│ Marrakech  │ Développeur Full Stack   │         1 │                1 │      100.0 │         50 │
│ Tanger     │ Développeur Full Stack   │         1 │                0 │        0.0 │         50 │
│ Fès        │ Développeur Full Stack   │         1 │                1 │      100.0 │         50 │
│ Marrakech  │ Développeur Full Stack   │         1 │                0 │        0.0 │         50 │
│ Tanger     │ Développeur Full Stack   │         1 │                0 │        0.0 │         50 │
│ Fès        │ Développeur Full Stack   │         1 │                1 │      100.0 │         50 │
│ Marrakech  │ Développeur Full Stack   │         1 │                1 │      100.0 │         50 │
│ Fès        │ Développeur Full Stack   │         1 │                0 │        0.0 │         50 │
│ Rabat      │ Développeur Full Stack   │         1 │                0 │        0.0 │         50 │
│ Marrakech  │ Développeur Full Stack   │         1 │                1 │      100.0 │         50 │
└────────────┴──────────────────────────┴───────────┴──────────────────┴────────────┴────────────┘
  856 rows (40 shown)                 use .last to show entire result                  6 columns

┌──────────────────────────┬───────────┬────────────┬─────────────┐
│          profil          │ nb_offres │ pct_remote │ pct_vs_casa │
│         varchar          │   int64   │   double   │   double    │
├──────────────────────────┼───────────┼────────────┼─────────────┤
│ Autre IT                 │        17 │       29.4 │        NULL │
│ Autre IT                 │        15 │       66.7 │        NULL │
│ Autre IT                 │        15 │       66.7 │        NULL │
│ Autre IT                 │        13 │       76.9 │        NULL │
│ Autre IT                 │        13 │       53.8 │        NULL │
│ Autre IT                 │        12 │       50.0 │        NULL │
│ Autre IT                 │        12 │       58.3 │        NULL │
│ Autre IT                 │        12 │       83.3 │        NULL │
│ Autre IT                 │        11 │       72.7 │        NULL │
│ Autre IT                 │        11 │       45.5 │        NULL │
│ Autre IT                 │        10 │       70.0 │        NULL │
│ Autre IT                 │         9 │       44.4 │        NULL │
│ Autre IT                 │         8 │       62.5 │        NULL │
│ Autre IT                 │         8 │       62.5 │        NULL │
│ Autre IT                 │         8 │       75.0 │        NULL │
│ Autre IT                 │         8 │       37.5 │        NULL │
│ Autre IT                 │         8 │       12.5 │        NULL │
│ Autre IT                 │         7 │       57.1 │        NULL │
│ Autre IT                 │         6 │       66.7 │        NULL │
│ Autre IT                 │         5 │       20.0 │        NULL │
│    ·                     │         · │         ·  │          ·  │
│    ·                     │         · │         ·  │          ·  │
│    ·                     │         · │         ·  │          ·  │
│ Développeur Full Stack   │         1 │      100.0 │        NULL │
│ Cybersécurité            │         1 │        0.0 │        NULL │
│ DevOps / SRE             │         1 │      100.0 │        NULL │
│ Data Analyst             │         1 │      100.0 │        NULL │
│ Développeur Full Stack   │         1 │        0.0 │        NULL │
│ Chef de Projet IT        │         1 │        0.0 │        NULL │
│ Chef de Projet IT        │         1 │        0.0 │        NULL │
│ Data Engineer            │         1 │        0.0 │        NULL │
│ Cybersécurité            │         1 │      100.0 │        NULL │
│ Data Analyst             │         1 │      100.0 │        NULL │
│ Développeur Full Stack   │         1 │        0.0 │        NULL │
│ Développeur Frontend     │         1 │      100.0 │        NULL │
│ Développeur Backend      │         1 │        0.0 │        NULL │
│ Data Scientist           │         1 │        0.0 │        NULL │
│ Chef de Projet IT        │         1 │        0.0 │        NULL │
│ Admin Systèmes & Réseaux │         1 │        0.0 │        NULL │
│ DevOps / SRE             │         1 │        0.0 │        NULL │
│ Data Analyst             │         1 │      100.0 │        NULL │
│ Data Engineer            │         1 │        0.0 │        NULL │
│ Chef de Projet IT        │         1 │        0.0 │        NULL │
└──────────────────────────┴───────────┴────────────┴─────────────┘
  131 rows (40 shown)                                   4 columns

In [ ]:

Casablanca concentre très largement les opportunités IT : pour tous les profils (Admin systèmes, Développement, Data, DevOps…), le nombre d’offres à 
Casablanca est systématiquement supérieur ou égal à celui des autres villes. On observe par exemple pour le profil "Admin Systèmes & Réseaux" des
volumes allant jusqu’à 11 offres à Casablanca, contre 6 à Rabat et moins ailleurs. Cela confirme le statut de Casablanca comme principal hub 
technologique du Maroc.

Rabat se place en deuxième position, avec des offres souvent deux à trois fois moins nombreuses qu’à Casablanca, mais devant Tanger et Marrakech. Pour
la plupart des profils techniques, Rabat représente une alternative crédible, avec un écart moins marqué sur certains métiers (ex: Développeur Full
Stack, Data Engineer).

Tanger, malgré un volume plus faible, montre une activité non négligeable, notamment sur les profils "Autre IT", "Développeur Full Stack" et "Data 
Analyst". Cependant, en volume absolu, Tanger reste loin derrière Casablanca : à peine une dizaine d’offres pour les métiers les plus représentés, 
contre plusieurs dizaines à Casablanca. Le télétravail est annoncé dans 20 à 80% des offres selon les cas, avec des taux parfois élevés à Tanger
(ex: 100% pour certains postes Full Stack).

Marrakech et Fès sont beaucoup plus marginaux : quelques offres éparses, principalement sur des profils généralistes ou du développement web.

Remarque sur les doublons : Les résultats affichent plusieurs lignes pour un même couple (ville, profil). Cela vient probablement d’une granularité 
plus fine (mois ou sous‑catégorie) dans la table offres_par_ville. Pour une analyse fiable, il faudrait agréger les offres par ville et profil (en 
sommant nb_offres). Malgré ce bruit, la tendance reste claire : Casablanca domine, Rabat suit, Tanger est troisième mais très loin.

In [2]:
# Agrège les offres par ville et profil 
df_ville = pd.read_parquet('data_lake_root/gold/offres_par_ville.parquet')
df_ville_agg = df_ville.groupby(['ville', 'profil'])['nb_offres'].sum().reset_index()
# Filtrer les 3 villes et 3 profils data
top_villes = ['Casablanca', 'Rabat', 'Tanger']
top_profils = ['Data Engineer', 'Data Analyst', 'Data Scientist']
df_filtre = df_ville_agg[df_ville_agg['ville'].isin(top_villes) & df_ville_agg['profil'].isin(top_profils)]

import seaborn as sns
plt.figure(figsize=(10,6))
sns.barplot(data=df_filtre, x='ville', y='nb_offres', hue='profil')
plt.title('Offres IT par ville et profil')
plt.ylabel('Nombre d’offres')
plt.savefig('offres_par_ville.png')
plt.show()

<class 'NameError'>: name 'pd' is not defined

In [ ]:
##Question 3 — Quel est le salaire médian par profil IT au Maroc ?

In [ ]:
### Résultats nationaux
┌──────────────────────────┬─────────────────┬───┬────────────────────┬──────────────────┬─────────────────┐
│          profil          │ nb_offres_total │ … │ salaire_median_mad │ salaire_plancher │ salaire_plafond │
│         varchar          │     int128      │ … │       double       │      double      │     double      │
├──────────────────────────┼─────────────────┼───┼────────────────────┼──────────────────┼─────────────────┤
│ DevOps / SRE             │              49 │ … │            15000.0 │           4000.0 │         55000.0 │
│ Développeur Full Stack   │             192 │ … │            13875.0 │           3000.0 │         55000.0 │
│ Architecte IT            │              78 │ … │            13500.0 │           3000.0 │         55000.0 │
│ Cybersécurité            │              83 │ … │            13125.0 │           3000.0 │         55000.0 │
│ Chef de Projet IT        │             157 │ … │            13000.0 │           3000.0 │         55000.0 │
│ Admin Systèmes & Réseaux │             266 │ … │            13000.0 │           3000.0 │         55000.0 │
│ Autre IT                 │            2769 │ … │            12500.0 │           3000.0 │         55000.0 │
│ Data Scientist           │             198 │ … │            12500.0 │           3000.0 │         55000.0 │
│ Développeur Frontend     │              77 │ … │            12500.0 │           3000.0 │         55000.0 │
│ Data Engineer            │              43 │ … │            12250.0 │           4000.0 │         55000.0 │
│ Data Analyst             │             214 │ … │            11000.0 │           3000.0 │         55000.0 │
│ Développeur Backend      │              85 │ … │            10500.0 │           3000.0 │         55000.0 │
└──────────────────────────┴─────────────────┴───┴────────────────────┴──────────────────┴─────────────────┘
  12 rows                       use .last to show entire result                        7 columns (5 shown)

##Interprétation:
Les profils les mieux rémunérés sont DevOps / SRE (15 000 MAD médian) et Développeur Full Stack (13 875 MAD). Ces métiers, très techniques et liés 
à l’automatisation, au cloud et aux environnements modernes, sont les plus valorisés sur le marché marocain.

Viennent ensuite Architecte IT, Cybersécurité, Chef de Projet IT et Admin Systèmes & Réseaux avec des médianes entre 13 000 et 13 500 MAD. Ces profils
bénéficient d’une bonne reconnaissance salariale, souvent liée à l’expérience et aux responsabilités.

Les métiers de la data (Data Scientist, Data Engineer) affichent des salaires médians de 12 500 et 12 250 MAD respectivement, légèrement en dessous des
profils DevOps. Le Data Analyst est moins bien rémunéré (11 000 MAD), ce qui correspond à des niveaux de compétences souvent moins techniques ou plus
débutants.

Les développeurs Backend (10 500 MAD) ferment la marche, probablement car les offres incluent des postes junior ou dans des technologies moins 
demandées.

On note que le salaire plancher est souvent à 3 000 MAD (stages ou très junior) et le plafond à 55 000 MAD (postes très senior ou experts). L’écart
est large, traduisant une forte hétérogénéité des expériences et des responsabilités.

In [ ]:
### Focus Tanger
┌──────────────────────────┬───────────┬────────────────────┬────────────────┬────────────────┬─────────────────────────┐
│          profil          │ nb_offres │ salaire_median_mad │ salaire_q1_mad │ salaire_q3_mad │ ecart_mediane_nationale │
│         varchar          │   int64   │       double       │     double     │     double     │         double          │
├──────────────────────────┼───────────┼────────────────────┼────────────────┼────────────────┼─────────────────────────┤
│ Autre IT                 │        15 │            20000.0 │         8250.0 │        23000.0 │                  7000.0 │
│ Autre IT                 │         6 │            18250.0 │        11000.0 │        27500.0 │                  5250.0 │
│ Développeur Backend      │         5 │            18000.0 │        14250.0 │        18000.0 │                     0.0 │
│ Chef de Projet IT        │        10 │            16500.0 │        11875.0 │        19750.0 │                     0.0 │
│ Data Scientist           │         9 │            15500.0 │        12375.0 │        17500.0 │                     0.0 │
│ Développeur Full Stack   │        10 │            13500.0 │         8125.0 │        15750.0 │                     0.0 │
│ Autre IT                 │        93 │            13250.0 │         7625.0 │        20000.0 │                   250.0 │
│ Autre IT                 │        36 │            13000.0 │         9000.0 │        20500.0 │                     0.0 │
│ Admin Systèmes & Réseaux │        12 │            11500.0 │        11000.0 │        14500.0 │                     0.0 │
│ Data Analyst             │        13 │            11000.0 │         6750.0 │        14000.0 │                     0.0 │
│ Autre IT                 │        21 │             7875.0 │         6938.0 │        13625.0 │                 -5125.0 │
│ Autre IT                 │        13 │             7750.0 │         6063.0 │        21125.0 │                 -5250.0 │
│ Autre IT                 │        35 │             3250.0 │         3250.0 │         3250.0 │                 -9750.0 │
└──────────────────────────┴───────────┴────────────────────┴────────────────┴────────────────┴─────────────────────────┘ 
13 rows 6 columns

##Interprétation:
À Tanger, les salaires médians sont globalement plus élevés que la médiane nationale pour plusieurs profils, notamment pour la catégorie "Autre IT" 
    (qui regroupe beaucoup d’offres). On observe des médianes à 20 000 MAD et 18 250 MAD, bien au‑dessus des 12 500 MAD nationaux. Cela peut 
s’expliquer par une présence plus forte d’entreprises internationales ou de postes à haute valeur ajoutée dans la région.

Les Développeurs Backend (18 000 MAD) et Chefs de Projet IT (16 500 MAD) à Tanger égalent ou dépassent leurs équivalents nationaux (respectivement
10 500 et 13 000 MAD). Les Data Scientists (15 500 MAD) et Développeurs Full Stack (13 500 MAD) sont également bien positionnés.

Cependant, la présence de multiples lignes pour "Autre IT" suggère que la table n’est pas suffisamment agrégée (par sous‑profil ou par ville).
Malgré ce bruit, la tendance est claire : Tanger propose des salaires compétitifs, parfois supérieurs à la moyenne nationale, ce qui en fait une
place attractive pour les talents IT.

In [ ]:

df_salaire = pd.read_parquet('data_lake_root/gold/salaires_par_profil.parquet')
# Agrège par profil (médiane nationale)
salaire_median = df_salaire.groupby('profil')['salaire_median_mad'].median().sort_values(ascending=False).head(12)

plt.figure(figsize=(12,6))
salaire_median.plot(kind='bar', color='coral')
plt.title('Salaire médian par profil IT (MAD)')
plt.ylabel('Salaire médian (MAD)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('salaires_par_profil.png')
plt.show()

In [ ]:

##Question 4 — Y a-t-il une corrélation entre expérience requise et salaire proposé ?

In [ ]:

┌──────────────────────────┬───────────────────────────────┬─────────────────────┐
│          profil          │ nb_offres_avec_salaire_et_exp │ correlation_pearson │
│         varchar          │             int64             │       double        │
├──────────────────────────┼───────────────────────────────┼─────────────────────┤
│ Data Engineer            │                            49 │               0.913 │
│ Chef de Projet IT        │                           137 │               0.901 │
│ Architecte IT            │                            78 │               0.898 │
│ Cybersécurité            │                            78 │               0.875 │
│ Développeur Frontend     │                            77 │               0.875 │
│ Autre IT                 │                          1636 │               0.871 │
│ Développeur Full Stack   │                           154 │               0.865 │
│ DevOps / SRE             │                            71 │               0.862 │
│ Data Analyst             │                           173 │               0.858 │
│ Admin Systèmes & Réseaux │                           207 │               0.855 │
│ Développeur Backend      │                            81 │               0.847 │
│ Data Scientist           │                           161 │               0.842 │
└──────────────────────────┴───────────────────────────────┴─────────────────────┘
  12 rows                                                              3 columns

In [ ]:

##Interprétation:
Les résultats montrent une corrélation positive très élevée entre le nombre d’années d’expérience minimal requis et le salaire médian proposé, 
pour l’ensemble des profils IT. Les coefficients de Pearson dépassent 0,84, atteignant même 0,913 pour les Data Engineers et 0,901 pour les Chefs 
de Projet IT. Cela signifie que plus une offre exige d’expérience, plus le salaire proposé est élevé, de manière presque linéaire.

Les Data Engineers affichent la corrélation la plus forte (0,913), ce qui indique que dans ce métier très technique, chaque année d’expérience
supplémentaire se traduit par une augmentation salariale très régulière. Les Chefs de Projet et Architectes suivent de près, confirmant que 
l’expérience est un facteur clé de rémunération dans les postes à responsabilités.

Même pour les profils avec une corrélation légèrement plus faible (comme Data Scientist à 0,842 ou Développeur Backend à 0,847), le lien reste très 
significatif. Aucun profil ne descend en dessous de 0,84, ce qui montre que sur le marché marocain, l’expérience est un déterminant universel du 
salaire.

Ces résultats sont cohérents avec la question 3 (salaires médians par profil). Par exemple, les DevOps et Data Engineers, qui sont parmi les mieux
payés, ont aussi une forte corrélation, car la progression salariale avec l’expérience est rapide. Les Data Analysts, bien que moins bien rémunérés
en médiane, bénéficient également d’une progression liée à l’expérience (corrélation 0,858).

In [ ]:

df_offres = pd.read_parquet('data_lake_root/silver/offres_clean/offres_clean.parquet')
# Filtrer les offres avec salaire et expérience
df_corr = df_offres[(df_offres['salaire_connu']) & (df_offres['experience_min_ans'].notna())]

plt.figure(figsize=(10,6))
sns.regplot(data=df_corr, x='experience_min_ans', y='salaire_median_mad', 
            scatter_kws={'alpha':0.3}, line_kws={'color':'red'})
plt.xlabel('Expérience minimale requise (années)')
plt.ylabel('Salaire médian (MAD)')
plt.title('Corrélation expérience – salaire')
plt.savefig('correlation_exp_salaire.png')
plt.show()

In [ ]:

##Question 5 — Quelles entreprises recrutent le plus ? Qui sont les concurrents de Mexora sur le marché du talent ?

In [ ]:

┌──────────────────────┬────────────┬────────────────────┬──────────────────────┬──────────────────────┬────────────────┐
│      entreprise      │   ville    │ nb_offres_publiees │ nb_profils_differen… │ salaire_moyen_propo… │ rang_recruteur │
│       varchar        │  varchar   │       int64        │        int64         │        double        │     int64      │
├──────────────────────┼────────────┼────────────────────┼──────────────────────┼──────────────────────┼────────────────┤
│ TechConsult Maroc    │ Casablanca │                 38 │                   11 │              16870.0 │              1 │
│ Expleo Maroc         │ Casablanca │                 38 │                    9 │              17467.0 │              1 │
│ Micropole Maroc      │ Casablanca │                 36 │                   10 │              14056.0 │              3 │
│ Sopra Steria Maroc   │ Casablanca │                 36 │                    9 │              12920.0 │              3 │
│ Aubay Maroc          │ Casablanca │                 34 │                    9 │              14391.0 │              5 │
│ Business & Decision… │ Casablanca │                 34 │                    9 │              15261.0 │              5 │
│ HPS Group            │ Casablanca │                 33 │                    8 │              14705.0 │              7 │
│ CapgeminiMaroc       │ Casablanca │                 32 │                   11 │              17250.0 │              8 │
│ Cognizant Casablanca │ Casablanca │                 32 │                    9 │              16781.0 │              8 │
│ Proxym Maroc         │ Casablanca │                 32 │                    7 │              14533.0 │              8 │
│ Accenture Maroc      │ Casablanca │                 31 │                    8 │              16172.0 │             11 │
│ M2M Group            │ Casablanca │                 31 │                    8 │              18762.0 │             11 │
│ FezSoft              │ Casablanca │                 31 │                    6 │              14630.0 │             11 │
│ SQLI Maroc           │ Casablanca │                 30 │                    5 │              17457.0 │             14 │
│ Inetum Maroc         │ Casablanca │                 30 │                   10 │              13029.0 │             14 │
│ Wavestone Maroc      │ Casablanca │                 29 │                    8 │              14868.0 │             16 │
│ Alten Maroc          │ Casablanca │                 28 │                    5 │              14400.0 │             17 │
│ Mazars Digital Maroc │ Casablanca │                 28 │                    6 │              20850.0 │             17 │
│ Solucom Maroc        │ Casablanca │                 28 │                    7 │              13861.0 │             17 │
│ Maroc Telecom IT     │ Casablanca │                 28 │                    8 │              16044.0 │             17 │

└──────────────────────┴────────────┴────────────────────┴──────────────────────┴──────────────────────┴────────────────┘  
20 rows    6 columns
##Interprétation:
Casablanca concentre l’intégralité du top 20, confirmant son statut de capitale économique et technologique. Les entreprises de conseil et ESN
(TechConsult, Expleo, Sopra Steria, Capgemini, Accenture, etc.) dominent largement le recrutement, avec des volumes de 28 à 38 offres chacune.
    Ces sociétés recrutent sur une grande diversité de profils (entre 5 et 11 métiers différents), ce qui montre leur besoin de polyvalence pour
répondre aux missions clients.

Les salaires moyens proposés varient de 12 920 MAD (Sopra Steria) à 20 850 MAD (Mazars Digital Maroc). Ce dernier se distingue par une rémunération
plus élevée, sans être le plus gros recruteur (28 offres). D’autres comme Expleo (17 467 MAD) et TechConsult (16 870 MAD) offrent des salaires
compétitifs.

Pour Mexora, ces entreprises sont des concurrents directs sur le marché casablancais. Pour se démarquer, Mexora pourrait miser sur des salaires
attractifs (au-dessus de 17 000 MAD) ou sur des avantages spécifiques (télétravail, formation, etc.). Les volumes de recrutement (30‑40 offres) 
indiquent une forte demande, ce qui est une opportunité pour Mexora si elle parvient à attirer les talents.

In [ ]:

┌────────────────┬────────────────────┬────────────────────────────────────┬───────────────────────┬────────────────────┐
│   entreprise   │ nb_offres_publiees │          profils_recrutes          │ salaire_moyen_propose │ niveau_competition │
│    varchar     │       int64        │             varchar[]              │        double         │      varchar       │
├────────────────┼────────────────────┼────────────────────────────────────┼───────────────────────┼────────────────────┤
│ FinStart Maroc │                 13 │ [Admin Systèmes & Réseaux,         │               13850.0 │ Compétiteur moyen  │
│                │                    │  Autre IT, Cybersécurité,          │                       │                    │
│                │                    │  Data Analyst, Data Scientist,     │                       │                    │
│                │                    │  Développeur Backend]              │                       │                    │
└────────────────┴────────────────────┴────────────────────────────────────┴───────────────────────┴────────────────────┘
##Interprétation:

Sur le plan national, les principaux concurrents de Mexora sont les grandes ESN et cabinets de conseil installés à Casablanca. TechConsult,
Expleo, Capgemini, Accenture, M2M Group et Mazars Digital sont les plus actifs, avec des volumes de recrutement compris entre 28 et 38 offres.
Ces entreprises recrutent une large palette de profils IT (de 5 à 11 métiers différents), ce qui montre une forte concurrence pour attirer
les talents.

En termes de rémunération, Mazars Digital se distingue avec un salaire moyen de 20 850 MAD, le plus élevé du top 20. M2M Group (18 762 MAD) et Expleo 
(17 467 MAD) offrent également des salaires attractifs. Mexora devra probablement s’aligner sur ces niveaux pour attirer les profils les plus recherchés
(Data Engineer, DevOps, etc.).

À Tanger, le marché est moins dense. FinStart Maroc est le seul concurrent identifié recrutant des profils Data (Analyst et Scientist) avec 13 offres
et un salaire moyen de 13 850 MAD, ce qui le place dans la catégorie « compétiteur moyen » (entre 12 000 et 20 000 MAD). Cela suggère que la concurrence
à Tanger est moins féroce qu’à Casablanca, offrant à Mexora une opportunité de s’implanter ou de renforcer sa présence avec des salaires légèrement 
supérieurs (par exemple 15 000‑16 000 MAD) pour les profils Data.

Cependant, l’absence d’autres entreprises dans les données peut indiquer soit une sous‑représentation des recruteurs tangérois dans le dataset,
soit un marché effectivement peu concurrentiel pour les métiers data. Mexora pourrait capitaliser sur ce vide en proposant des packages attractifs
(télétravail, formation, avantages) pour devenir un employeur de référence à Tanger

In [ ]:
df_ent = pd.read_parquet('data_lake_root/gold/entreprises_recruteurs.parquet')
top10 = df_ent.nlargest(10, 'nb_offres_publiees')[['entreprise', 'nb_offres_publiees']].sort_values('nb_offres_publiees')

plt.figure(figsize=(10,6))
top10.plot(kind='barh', x='entreprise', y='nb_offres_publiees', legend=False, color='green')
plt.xlabel('Nombre d’offres publiées')
plt.title('Top 10 entreprises recruteuses (IT Maroc)')
plt.tight_layout()
plt.savefig('top_entreprises.png')
plt.show()